# Pre-entrega: Pipeline de entrenamiento, validación y clasificador base
**Problema:** dado el título de una publicación de Mercado Libre, predecir su categoría (8 clases).

**Antes de empezar:** en Colab, `Entorno de ejecución > Cambiar tipo de entorno > GPU (T4)` es opcional; con CPU también entrena en segundos.

## 1. Configuración del entorno

In [ ]:
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset

print("Versión de PyTorch:", torch.__version__)

In [ ]:
@dataclass
class Config:
    seed: int = 42
    val_size: float = 0.2
    max_features: int = 5000
    hidden_dim: int = 128
    dropout: float = 0.3
    batch_size: int = 64
    epochs: int = 30
    learning_rate: float = 1e-3

cfg = Config()


def get_device() -> torch.device:
    """Detecta automáticamente el mejor dispositivo: CUDA > MPS > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def set_seed(seed: int) -> None:
    """Fija semillas de Python, NumPy y PyTorch para reproducibilidad."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(cfg.seed)
device = get_device()
print("Dispositivo:", device)

## 2. Datos
Subí `dataset.csv` (columnas `id_base`, `titulo`, `categoria`). La celda lo pide solo si no está en `/content`.

In [ ]:
import os

DATA_PATH = "/content/dataset.csv"
if not os.path.exists(DATA_PATH):
    from google.colab import files
    uploaded = files.upload()          # elegir dataset.csv
    name = next(iter(uploaded))
    os.rename(name, DATA_PATH)

df = pd.read_csv(DATA_PATH).dropna(subset=["titulo", "categoria"])
print(df.shape)
df.sample(5, random_state=0)

In [ ]:
df["categoria"].value_counts().plot(kind="barh", title="Títulos por clase")
plt.gca().invert_yaxis()
plt.show()

## 3. Split train/validación y vectorización
Split **estratificado** 80/20. El TF-IDF se ajusta **solo con train** para evitar fuga de información. Los pesos por clase compensan el desbalance.

In [ ]:
classes = sorted(df["categoria"].unique())
label2idx = {c: i for i, c in enumerate(classes)}
y = df["categoria"].map(label2idx).to_numpy()

x_tr_txt, x_val_txt, y_tr, y_val = train_test_split(
    df["titulo"].to_numpy(), y,
    test_size=cfg.val_size, stratify=y, random_state=cfg.seed,
)

vectorizer = TfidfVectorizer(
    max_features=cfg.max_features, ngram_range=(1, 2), min_df=2,
    strip_accents="unicode", sublinear_tf=True,
)
x_tr = vectorizer.fit_transform(x_tr_txt).toarray().astype(np.float32)
x_val = vectorizer.transform(x_val_txt).toarray().astype(np.float32)

weights = compute_class_weight("balanced", classes=np.arange(len(classes)), y=y_tr)
class_weights = torch.tensor(weights, dtype=torch.float32)

train_ds = TensorDataset(torch.from_numpy(x_tr), torch.from_numpy(y_tr).long())
val_ds = TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val).long())

g = torch.Generator().manual_seed(cfg.seed)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, generator=g)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)

input_dim = x_tr.shape[1]
print(f"Train: {len(train_ds)} | Validación: {len(val_ds)} | Features: {input_dim} | Clases: {len(classes)}")

## 4. Arquitectura base (MLP)

In [ ]:
class MLPClassifier(nn.Module):
    """input_dim -> hidden_dim -> n_classes. Devuelve logits (sin softmax)."""

    def __init__(self, input_dim, hidden_dim, n_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, x):
        return self.net(x)


model = MLPClassifier(input_dim, cfg.hidden_dim, len(classes), cfg.dropout).to(device)
print(model)
print("Parámetros:", sum(p.numel() for p in model.parameters()))

## 5. Ciclo de entrenamiento y validación

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()           # 1. reiniciar gradientes acumulados
        logits = model(xb)              # 2. forward pass
        loss = criterion(logits, yb)    # 3. cálculo de la pérdida
        loss.backward()                 # 4. backward pass (autograd)
        optimizer.step()                # 5. actualización de pesos con Adam

        running_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += xb.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evalúa en datos no vistos: sin gradientes y con el modelo en modo eval."""
    model.eval()
    running_loss, total = 0.0, 0
    y_true, y_pred = [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)

        running_loss += loss.item() * xb.size(0)
        total += xb.size(0)
        y_true.extend(yb.cpu().tolist())
        y_pred.extend(logits.argmax(dim=1).cpu().tolist())

    metrics = {
        "loss": running_loss / total,
        "accuracy": sum(t == p for t, p in zip(y_true, y_pred)) / total,
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
    }
    return metrics, y_true, y_pred

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)

history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "val_f1"]}
best_val_loss, best_state = float("inf"), None

for epoch in range(1, cfg.epochs + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val, _, _ = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(val["loss"])
    history["val_acc"].append(val["accuracy"])
    history["val_f1"].append(val["f1_macro"])

    if val["loss"] < best_val_loss:
        best_val_loss = val["loss"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(f"Época {epoch:02d}/{cfg.epochs} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
          f"val loss {val['loss']:.4f} acc {val['accuracy']:.3f} f1 {val['f1_macro']:.3f}")

## 6. Curvas de pérdida y accuracy

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(epochs, history["train_loss"], label="train")
ax1.plot(epochs, history["val_loss"], label="validación")
ax1.set(title="Pérdida por época", xlabel="Época", ylabel="CrossEntropy")
ax1.legend()

ax2.plot(epochs, history["train_acc"], label="train")
ax2.plot(epochs, history["val_acc"], label="validación")
ax2.set(title="Accuracy por época", xlabel="Época", ylabel="Accuracy")
ax2.legend()

fig.tight_layout()
fig.savefig("curves.png", dpi=150)
plt.show()

## 7. Evaluación final con el mejor modelo (menor pérdida de validación)

In [ ]:
model.load_state_dict(best_state)
val, y_true, y_pred = evaluate(model, val_loader, criterion, device)
print(f"Val loss {val['loss']:.4f} | Accuracy {val['accuracy']:.3f} | F1 macro {val['f1_macro']:.3f}\n")
print(classification_report(y_true, y_pred, target_names=classes, digits=3))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(classes)), classes, rotation=45, ha="right")
ax.set_yticks(range(len(classes)), classes)
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set(title="Matriz de confusión (validación)", xlabel="Predicho", ylabel="Real")
fig.tight_layout()
plt.show()

## 8. Datos para el README

In [ ]:
print("Versión de PyTorch:", torch.__version__)
print("Dispositivo:", device)
print("learning_rate:", cfg.learning_rate)
print(f"Pérdida train: {history['train_loss'][0]:.4f} -> {history['train_loss'][-1]:.4f}")
print(f"Pérdida val:   {history['val_loss'][0]:.4f} -> {history['val_loss'][-1]:.4f}")
print("Mejor época (val loss):", int(np.argmin(history["val_loss"])) + 1)

In [ ]:
# Descargar la gráfica y el historial para subirlos al repo
pd.DataFrame(history).to_csv("history.csv", index_label="epoch")
try:
    from google.colab import files
    files.download("curves.png")
    files.download("history.csv")
except ImportError:
    pass

### Interpretación de la curva de pérdida
*(Completar tras ejecutar: ¿bajó la pérdida de train y de validación? ¿en qué época se estabiliza? ¿hay señales de overfitting, es decir, val loss que sube mientras train loss sigue bajando? ¿qué clases se confunden según la matriz?)*